# IMPORT ALL NECESSARY PACKAGES IN THIS CELL

In [ ]:
# =============================================================================
# IMPORTS — portable: works from any directory this folder lives in
# =============================================================================
# As long as this notebook is in the SAME folder as the .py files, this cell
# finds them automatically. No pip install, no path hardcoding.

import sys, os

try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
if _here not in sys.path:
    sys.path.insert(0, _here)

from Calendar import (
    CalendarManager, CalendarType, load_all_calendars, print_calendar_summary,
)
from workforce import (
    DeterministicWorkforce, WorkerGroup, Months,
)
from inventory import (
    DeterministicInventory, DeterministicDemand, DemandStream, AgeDistributionStream,
)
from inventory_parser import (
    inventory_from_paste, inventory_from_input_or_paste, print_inventory_summary,
)
from Cycle_time_calculator import (
    DeterministicScenario, DeterministicCycleTimeCalculator,
)
from diagnostics import (
    weekly_summary, compare_scenarios_table,
    monthly_candlestick_chart, daily_inventory_with_p_chart, monthly_waterfall_chart,
    render_scenario_charts,
)
from runner import run_scenarios

from datetime import date, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("All modules imported ✅")

# Inputs

Select one or more queues via `QUEUES = "LE"` or `QUEUES = "LE, EDD"`. Each
queue has its own config block below. Paste zones for queues whose inventory
comes from raw extracts are at the top — clearly marked.

In [ ]:
# ============================================================================
# QUEUE SELECTION — two patterns, pick one:
#
# Pattern A (simple): comma-separated list. Runs each queue once.
#     QUEUES = "LE"           or  "LE, EDD"
#
# Pattern B (advanced): SCENARIOS dict for N named scenarios (same or
#   different queues, with different overrides). Runs all of them.
#     SCENARIOS = {
#         "LE baseline":        build_scenario("LE"),
#         "LE high attrition":  build_scenario("LE", overrides={"attrition_per_month": 2.0}),
#         "EDD as-is":          build_scenario("EDD"),
#     }
#
# Use QUEUES to fill SCENARIOS automatically, or define SCENARIOS yourself.
# If BOTH are set, SCENARIOS wins.
# ============================================================================
QUEUES = "LE"
SCENARIOS = None   # set to a dict to override pattern A

# ============================================================================
# SIMULATION PERIOD
# ============================================================================
start_date = date.fromisoformat("2026-01-06")
end_date   = date.fromisoformat("2026-12-31")

# ============================================================================
# PASTE ZONES — raw inventory extracts for queues that use parse-from-paste
# ============================================================================

EDD_INVENTORY_RAW = """
QUEUE	ALERT_TYPE	Days Difference	STATE
EDD	Parent Alert	93	Open
EDD	Parent Alert	107	Open
EDD	Parent Alert	114	Open
EDD	Parent Alert	50	Open
EDD	Parent Alert	70	Open
"""
# ^^ REPLACE with your actual EDD extract. Any columns; parser keeps what it needs.

# ---- LE inventory paste (alerts table + cases table stacked, OPTIONAL) ----
# LE currently uses the inline `inv_input` in build_scenario("LE"). If you
# paste here too, this paste WINS (paste-vs-inline precedence — see helper
# inventory_from_input_or_paste). Leave empty to keep using the inline string.
LE_INVENTORY_RAW = """
"""

# ---- AC inventory paste (alerts table + cases table stacked) ----
# Paste the ALERTS extract first, then the CASES extract directly below it.
# Each has its own header row. Parser auto-detects both and merges them
# into a single FIFO pool.
AC_INVENTORY_RAW = """
QUEUE\tALERT_TYPE\tDays Difference\tSTATE
AC\tAlert\t5\tOpen
AC\tAlert\t15\tOpen
AC\tAlert\t20\tOpen
QUEUE\tCASE_TYPE\tcase age\tSTATE
AC\tCase\t12\tOpen
AC\tCase\t30\tOpen
"""
# ^^ REPLACE with your actual AC extracts (alerts on top, cases underneath).

# ---- SIU inventory paste (alerts table + cases table stacked) ----
SIU_INVENTORY_RAW = """
QUEUE\tALERT_TYPE\tDays Difference\tSTATE
SIU\tAlert\t8\tOpen
SIU\tAlert\t15\tOpen
QUEUE\tCASE_TYPE\tcase age\tSTATE
SIU\tCase\t45\tOpen
SIU\tCase\t62\tOpen
"""
# ^^ REPLACE with your actual SIU extracts (alerts on top, cases underneath).

# ============================================================================
# SHARED — calendar
# ============================================================================
calendar_manager = CalendarManager()
sim_years = list(range(start_date.year, end_date.year + 1))
load_all_calendars(calendar_manager, sim_years)

# ============================================================================
# build_scenario(queue_name, overrides=None) — single source for all queue
# configs. The `overrides` dict lets you tweak WorkerGroup fields for
# alternate scenarios (e.g. {"attrition_per_month": 2.0} applied to every
# internal group). Keep simple for now; extend as needs grow.
# ============================================================================

def build_scenario(queue_name: str, overrides: dict = None) -> DeterministicScenario:
    overrides = overrides or {}

    # ---------------- LE ----------------
    if queue_name == "LE":
        LE_NON_PROD = Months(
            jan=0.7455, feb=0.7316, mar=0.7380, apr=0.7529,
            may=0.8770, jun=0.6091, jul=0.6961, aug=0.6504,
            sep=0.7066, oct=0.6743, nov=0.6139, dec=0.5827,
        )
        DEFAULT_TPT = 2.4

        worker_groups = [
            WorkerGroup(
                calendar_type=CalendarType.INTERNAL, name="Internal US",
                current_headcount=45.8, ramp_period_months=6,
                attrition_per_month=overrides.get("attrition_per_month", 1.0),
                tpt_by_month=overrides.get("tpt_by_month", DEFAULT_TPT),
                fte_conversion_by_month=LE_NON_PROD,
            ),
            WorkerGroup(
                calendar_type=CalendarType.USA, name="RightSource US",
                current_headcount=5.9, ramp_period_months=4,
                tpt_by_month=overrides.get("tpt_by_month", DEFAULT_TPT),
            ),
        ]

        demand = DeterministicDemand(streams=[
            DemandStream(
                name="314a", cadence="daily",
                monthly_volume=Months(
                    jan=5.4, feb=6.0, mar=5.4, apr=5.6, may=5.5, jun=5.7,
                    jul=5.5, aug=5.5, sep=5.7, oct=5.5, nov=5.7, dec=5.6,
                ),
                arrival_age=1,
            ),
            DemandStream(
                name="subpoena", cadence="on_days_of_month",
                days_of_month=[13, 27],
                monthly_volume=Months(
                    jan=1502, feb=1580, mar=1504, apr=1508, may=1512, jun=1516,
                    jul=1518, aug=1522, sep=1526, oct=1530, nov=1534, dec=1538,
                ),
                arrival_age=1,
            ),
        ])

        # Inventory: TWO ways to give it. If you paste in LE_INVENTORY_RAW above,
        # the paste takes precedence over the inline string below.
        inv_input = (
            "1:2, 2:0, 3:0, 4:1, 5:0, 6:0, 7:1, 8:1, 9:554, 10:1, 11:0, 12:3, "
            "13:2, 14:2, 15:29, 16:0, 17:0, 18:5, 19:1, 20:2, 21:2, 22:28, "
            "23:699, 24:0, 25:0, 26:3, 27:6, 28:53, 29:0, 30:0, 31:0"
        )
        initial_inventory = inventory_from_input_or_paste(
            queue="LE",
            snapshot_date=start_date,
            inv_input=inv_input,
            inventory_raw=LE_INVENTORY_RAW,
        )

        return DeterministicScenario(
            name=queue_name,
            workforce=DeterministicWorkforce(worker_groups=worker_groups),
            initial_inventory=initial_inventory,
            demand=demand,
            calendar_manager=calendar_manager,
            start_date=start_date, end_date=end_date,
            open_inventory_ratio=2.49, closed_inventory_ratio=1.29,
            reporting_percentile=90,
            workable_age_min=None, workable_age_max=None,
        )

    # ---------------- EDD ----------------
    elif queue_name == "EDD":
        # TODO: user to fill in actual EDD non-prod factors + headcounts + volumes
        EDD_NON_PROD = Months(
            jan=0.75, feb=0.75, mar=0.75, apr=0.75, may=0.75, jun=0.75,
            jul=0.75, aug=0.75, sep=0.75, oct=0.75, nov=0.75, dec=0.75,
        )
        DEFAULT_TPT = 2.4

        worker_groups = [
            WorkerGroup(
                calendar_type=CalendarType.INTERNAL, name="EDD Internal",
                current_headcount=0.0,          # TODO
                ramp_period_months=6,
                attrition_per_month=overrides.get("attrition_per_month", 1.0),
                tpt_by_month=DEFAULT_TPT,
                fte_conversion_by_month=EDD_NON_PROD,
            ),
            WorkerGroup(
                calendar_type=CalendarType.USA, name="EDD RightSource US",
                current_headcount=0.0,          # TODO
                ramp_period_months=4,
                tpt_by_month=DEFAULT_TPT,
            ),
        ]

        demand = DeterministicDemand(streams=[
            DemandStream(
                name="alerts", cadence="on_weekdays",
                weekdays=[1],                   # Tuesday
                monthly_volume=Months(),        # TODO: per-Tuesday values per month
                arrival_age=1,
            ),
        ])

        # Inventory: paste in EDD_INVENTORY_RAW above OR fill inv_input below.
        # If both, the paste wins.
        inv_input = ""    # e.g. "65:1, 70:1, 75:1, 100:5"
        initial_inventory = inventory_from_input_or_paste(
            queue="EDD",
            snapshot_date=start_date,
            inv_input=inv_input,
            inventory_raw=EDD_INVENTORY_RAW,
        )

        return DeterministicScenario(
            name=queue_name,
            workforce=DeterministicWorkforce(worker_groups=worker_groups),
            initial_inventory=initial_inventory,
            demand=demand,
            calendar_manager=calendar_manager,
            start_date=start_date, end_date=end_date,
            open_inventory_ratio=1.0, closed_inventory_ratio=1.0,   # TODO
            reporting_percentile=95,
            workable_age_min=61, workable_age_max=None,
        )

    # ---------------- AC ----------------
    elif queue_name == "AC":
        # Same worker-group structure as LE/EDD: Internal + RS_USA.
        # Single daily demand stream (alerts + cases combined).
        # No workable-age window (everything in inventory is workable).
        # Reporting percentile: 95.

        # TODO: user to fill in actual AC non-prod factors per month
        AC_NON_PROD = Months(
            jan=0.75, feb=0.75, mar=0.75, apr=0.75, may=0.75, jun=0.75,
            jul=0.75, aug=0.75, sep=0.75, oct=0.75, nov=0.75, dec=0.75,
        )
        DEFAULT_TPT = 2.4  # TODO: confirm AC TPT

        worker_groups = [
            WorkerGroup(
                calendar_type=CalendarType.INTERNAL, name="AC Internal",
                current_headcount=0.0,          # TODO: fill in
                ramp_period_months=6,
                attrition_per_month=overrides.get("attrition_per_month", 1.0),
                tpt_by_month=DEFAULT_TPT,
                fte_conversion_by_month=AC_NON_PROD,
            ),
            WorkerGroup(
                calendar_type=CalendarType.USA, name="AC RightSource US",
                current_headcount=0.0,          # TODO: fill in
                ramp_period_months=4,
                tpt_by_month=DEFAULT_TPT,
            ),
        ]

        # Single stream for now (alerts + cases combined).
        #
        # ⚠️  arrival_age=1 is an UNVERIFIED ASSUMPTION for cases.
        #    If cases actually enter the queue already aged (e.g. batched from
        #    upstream intake), change arrival_age here. See TODO.md §0.
        demand = DeterministicDemand(streams=[
            DemandStream(
                name="demand",
                cadence="daily",
                monthly_volume=Months(),        # TODO: per-day volume per month
                arrival_age=1,                  # ⚠️  unverified for cases
            ),
        ])

        # Inventory: paste in AC_INVENTORY_RAW above OR fill inv_input below.
        # If both, the paste wins.
        inv_input = ""    # e.g. "5:3, 12:1, 20:2"
        initial_inventory = inventory_from_input_or_paste(
            queue="AC",
            snapshot_date=start_date,
            inv_input=inv_input,
            inventory_raw=AC_INVENTORY_RAW,
        )

        return DeterministicScenario(
            name=queue_name,
            workforce=DeterministicWorkforce(worker_groups=worker_groups),
            initial_inventory=initial_inventory,
            demand=demand,
            calendar_manager=calendar_manager,
            start_date=start_date, end_date=end_date,
            open_inventory_ratio=1.0,           # TODO: user to provide
            closed_inventory_ratio=1.0,         # TODO: user to provide
            reporting_percentile=95,
            workable_age_min=None,
            workable_age_max=None,
        )

    # ---------------- SIU ----------------
    elif queue_name == "SIU":
        # Worker groups: Internal + USA RightSource (same as LE/EDD/AC).
        # Single FIFO pool (alerts + cases merged via two age columns).
        # No workable-age window.
        # Reporting percentile: 90.
        # DEMAND DESIGN: deferred — user said "demand is the tougher part, come
        # back later". Placeholder single daily stream with empty volumes.

        # TODO: user to fill in actual SIU non-prod factors per month
        SIU_NON_PROD = Months(
            jan=0.75, feb=0.75, mar=0.75, apr=0.75, may=0.75, jun=0.75,
            jul=0.75, aug=0.75, sep=0.75, oct=0.75, nov=0.75, dec=0.75,
        )
        DEFAULT_TPT = 2.4  # TODO: confirm SIU TPT

        worker_groups = [
            WorkerGroup(
                calendar_type=CalendarType.INTERNAL, name="SIU Internal",
                current_headcount=0.0,          # TODO: fill in
                ramp_period_months=6,
                attrition_per_month=overrides.get("attrition_per_month", 1.0),
                tpt_by_month=DEFAULT_TPT,
                fte_conversion_by_month=SIU_NON_PROD,
            ),
            WorkerGroup(
                calendar_type=CalendarType.USA, name="SIU RightSource US",
                current_headcount=0.0,          # TODO: fill in
                ramp_period_months=4,
                tpt_by_month=DEFAULT_TPT,
            ),
        ]

        # SIU has three demand streams, each with its own ingestion age:
        #
        #   1. Alerts        → daily arrivals, arrival_age=8
        #   2. Cases         → daily arrivals, arrival_age=60
        #   3. TMO referrals → daily arrivals spread across MULTIPLE ages
        #                       per day (an age-distribution stream).
        #
        # All `arrival_age` values are user inputs — change below as needed.
        demand = DeterministicDemand(streams=[
            # ----- 1) Daily alerts arriving at age 8 -----
            DemandStream(
                name="alerts",
                cadence="daily",
                monthly_volume=Months(),         # TODO: daily alert volume per month
                arrival_age=8,                   # change if alerts arrive at a different age
            ),

            # ----- 2) Daily cases arriving at age 60 -----
            DemandStream(
                name="cases",
                cadence="daily",
                monthly_volume=Months(),         # TODO: daily case volume per month
                arrival_age=60,                  # change if cases arrive at a different age
            ),

            # ----- 3) TMO-referred cases (distribution of ages per day) -----
            # Keys are calendar months; values are {age_in_days: items_per_day}.
            # Example for Jan: {15: 1, 30: 2, 45: 1} means every day of January,
            # 1 item arrives at age 15, 2 at age 30, 1 at age 45.
            AgeDistributionStream(
                name="tmo_referrals",
                monthly_distribution={
                    # TODO: fill per-month distributions, e.g.:
                    # 1:  {15: 1, 30: 2, 45: 1},
                    # 2:  {15: 2, 30: 3, 45: 1},
                    # ...
                },
            ),
        ])

        # Inventory: paste in SIU_INVENTORY_RAW above (alerts table + cases table
        # stacked) OR fill inv_input below. If both, the paste wins.
        inv_input = ""    # e.g. "8:1, 15:1, 45:1, 62:1"
        initial_inventory = inventory_from_input_or_paste(
            queue="SIU",
            snapshot_date=start_date,
            inv_input=inv_input,
            inventory_raw=SIU_INVENTORY_RAW,
        )

        return DeterministicScenario(
            name=queue_name,
            workforce=DeterministicWorkforce(worker_groups=worker_groups),
            initial_inventory=initial_inventory,
            demand=demand,
            calendar_manager=calendar_manager,
            start_date=start_date, end_date=end_date,
            open_inventory_ratio=1.0,           # TODO: user to provide
            closed_inventory_ratio=1.0,         # TODO: user to provide
            reporting_percentile=90,
            workable_age_min=None,
            workable_age_max=None,
        )

    # ---------------- Fraud / ML (placeholder) ----------------
    elif queue_name in {"Fraud", "ML"}:
        raise NotImplementedError(
            f"{queue_name} not yet configured. No default values set yet. See TODO.md."
        )

    else:
        raise ValueError(
            f"Unknown queue {queue_name!r}. Valid: LE, SIU, EDD, AC, Fraud, ML."
        )

# ============================================================================
# BUILD SCENARIOS — either via SCENARIOS dict (Pattern B) or QUEUES (Pattern A)
# ============================================================================
if SCENARIOS is None:
    queue_list = [q.strip() for q in QUEUES.split(",") if q.strip()]
    SCENARIOS = {q: build_scenario(q) for q in queue_list}

# ============================================================================
# PRE-SIM VERIFICATION — calendar + workforce + scenario.validate()
# ============================================================================
print_calendar_summary(calendar_manager, sim_years)
for name, scenario in SCENARIOS.items():
    print(f"\n{'#'*72}\n## {name}\n{'#'*72}")
    scenario.print_validate()
    scenario.workforce.print_summary(start_date, end_date)
    print_inventory_summary(
        scenario.initial_inventory,
        workable_age_min=scenario.workable_age_min,
        workable_age_max=scenario.workable_age_max,
    )

# Run simulations

Runs `calculator.calculate()` for each scenario in `scenarios`. Prints monthly
metrics per queue at the configured percentile (LE/SIU=90, EDD/AC=95).

In [ ]:
# ============================================================================
# Run every scenario in SCENARIOS. Uses runner.run_scenarios which validates
# first and skips broken scenarios. Returns {name: (calculator, result)}.
# ============================================================================
all_results = run_scenarios(SCENARIOS)

# ============================================================================
# Per-scenario monthly metrics
# ============================================================================
for name, (calc, result) in all_results.items():
    print(f"\n{'='*72}\nMONTHLY METRICS: {name}\n{'='*72}")
    scenario = calc.scenario
    months = []
    d = scenario.start_date
    while d <= scenario.end_date:
        k = (d.year, d.month)
        if k not in months:
            months.append(k)
        d = date(d.year + 1, 1, 1) if d.month == 12 else date(d.year, d.month + 1, 1)

    rows = []
    for y, m in months:
        mm = calc.calculate_monthly_metrics(result, m, y)
        mm["year"] = y
        mm["month"] = m
        rows.append(mm)
    df = pd.DataFrame(rows)
    cols = ["year", "month", "percentile",
            "p_direct", "p_from_open_ratio", "p_from_closed_ratio"]
    print(df[cols].to_string(index=False, float_format=lambda x: f"{x:,.2f}" if pd.notna(x) else ""))

# Diagnostics

Per-worker-group monthly breakdown + daily totals, printed per queue.

In [ ]:
pd.options.display.float_format = "{:,.2f}".format

for queue_name, (calc, result) in all_results.items():
    print(f"\n{'='*72}\nDIAGNOSTICS: {queue_name}\n{'='*72}")

    group_rows, total_rows, arrival_rows = [], [], []
    for dr in result.daily_results:
        total_rows.append({
            "Date":           dr.date,
            "Year":           dr.date.year,
            "Month":          dr.date.month,
            "Total_Capacity": round(dr.total_capacity, 2),
            "Items_Closed":   dr.total_burned,
            "Inv_Count":      dr.open_inventory_after.get_total_items(),
            "Avg_Inv_Age":    round(dr.open_inventory_after.calculate_average_age(), 2),
        })
        for gs in dr.group_stats:
            group_rows.append({
                "Date":          dr.date,
                "Year":          dr.date.year,
                "Month":         dr.date.month,
                "Group":         gs["name"],
                "Day_Type":      gs["day_type"],
                "FTE":           round(gs["fte_for_month"], 2),
                "Capacity":      round(gs["capacity"], 2),
            })
        for a in dr.arrivals:
            arrival_rows.append({
                "Date":        dr.date,
                "Year":        dr.date.year,
                "Month":       dr.date.month,
                "Stream":      a["name"],
                "Volume":      int(a["volume"]),
                "Arrival_Age": a["arrival_age"],
            })

    group_df   = pd.DataFrame(group_rows)
    total_df   = pd.DataFrame(total_rows)
    arrival_df = pd.DataFrame(arrival_rows)

    print("\n-- Monthly per worker group --")
    per_group = group_df.groupby(["Year", "Month", "Group"]).agg(
        Avg_FTE=("FTE", "mean"),
        Total_Capacity=("Capacity", "sum"),
    ).reset_index()
    print(per_group.to_string(index=False))

    print("\n-- Monthly totals --")
    monthly_totals = total_df.groupby(["Year", "Month"]).agg(
        Total_Closed=("Items_Closed", "sum"),
        End_Month_Inv=("Inv_Count", "last"),
        Avg_Inv_Age=("Avg_Inv_Age", "mean"),
    ).reset_index()
    print(monthly_totals.to_string(index=False))

    if not arrival_df.empty:
        print("\n-- Monthly arrivals by stream --")
        arr_monthly = arrival_df.groupby(["Year", "Month", "Stream"]).agg(
            Total_Volume=("Volume", "sum"),
        ).reset_index()
        print(arr_monthly.to_string(index=False))

# Weekly diagnostics

Weekly rollup (ISO week) of capacity, burn, arrivals, inventory.

In [ ]:
for name, (calc, result) in all_results.items():
    print(f"\n{'='*72}\n  WEEKLY: {name}\n{'='*72}")
    wk = weekly_summary(result, calc)
    print(wk.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

# Per-scenario charts

For **every scenario** (whether you ran 1 or 6 queues), two charts render:

1. **Monthly inventory candlestick** — one candle per month: body = open→close,
   wicks = peak/trough during the month. Green = inventory dropped (progress),
   red = grew (fell behind). Pn badge at the top of each candle.

2. **Daily open inventory line** — same data, daily resolution. Monthly Pn
   labels above each month.

Pn defaults to the open-ratio method; the percentile (P90 vs P95) comes from
the scenario's `reporting_percentile`.

In [ ]:
# Renders BOTH charts for every scenario in all_results automatically.
# Works for 1 or N scenarios — the loop handles any count.
for name, (calc, result) in all_results.items():
    print(f"\n{'='*72}\n  CHARTS: {name}\n{'='*72}")
    render_scenario_charts(result, calc)  # produces 2 figures back-to-back

# Scenario comparison

Side-by-side table + overlay chart. Only useful when SCENARIOS has > 1 entry.

In [ ]:
if len(all_results) > 1:
    print("Scenario comparison — monthly metrics")
    cmp_df = compare_scenarios_table(all_results)
    print(cmp_df.to_string(index=False, float_format=lambda x: f'{x:,.2f}'))
else:
    print("Only one scenario ran — comparison view requires 2 or more.")